# Lecture 14. ETLN price forecasting: LSTM + CNN + CNN-LSTM


## 1. Install libraries

Run this in Colab or local environment.


In [ ]:
# For Google Colab (skip this cell for local run)
# !pip -q install tinkoff-investments tensorflow scikit-learn pandas matplotlib


## 2. Imports and settings


In [ ]:
import os
import json
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, roc_auc_score

from tinkoff.invest import Client, CandleInterval
from tinkoff.invest.utils import now

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

TOKEN = os.environ['TINVEST_TOKEN']
APP_NAME = 'lecture14-etln-colab'
FIGI = 'BBG00RM6M4T8'

DAYS_BACK = 1460
WINDOW = 20
FORECAST_HORIZON = 3
RANDOM_SEED = 42

OUTPUT_DIR = Path('reports/lecture14_cnn_lstm_etln_h3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def q_to_float(q):
    return float(q.units + q.nano / 1e9)


## 3. Download ETLN daily candles


In [ ]:
rows = []

with Client(TOKEN, app_name=APP_NAME) as api:
    for candle in api.get_all_candles(
        figi=FIGI,
        from_=now() - timedelta(days=DAYS_BACK),
        to=now(),
        interval=CandleInterval.CANDLE_INTERVAL_DAY,
    ):
        rows.append({'Date': candle.time, 'Close': q_to_float(candle.close)})

df = pd.DataFrame(rows)
df['Date'] = pd.to_datetime(df['Date'], utc=True)
df = df.drop_duplicates('Date').sort_values('Date').set_index('Date')
df.tail()


## 4. Plot close price


In [ ]:
ax = df['Close'].plot(title='ETLN close price (daily candles)')
ax.set_ylabel('RUB')
plt.show()


## 5. Build windows for models


In [ ]:
data = df[['Close']].copy()
values = data['Close'].astype('float32').values
dates = data.index

X_all, y_price_all, idx_all = [], [], []
for i in range(WINDOW, len(values) - FORECAST_HORIZON + 1):
    X_all.append(values[i - WINDOW:i])
    target_pos = i + FORECAST_HORIZON - 1
    y_price_all.append(values[target_pos])
    idx_all.append(dates[target_pos])

X_all = np.asarray(X_all, dtype='float32')
y_price_all = np.asarray(y_price_all, dtype='float32')
idx_all = pd.DatetimeIndex(idx_all)

n = len(data)
train_size = int(n * 0.70)
valid_size = int(n * 0.15)
valid_end = train_size + valid_size

train_boundary = data.index[train_size]
valid_boundary = data.index[valid_end]

train_mask = idx_all < train_boundary
valid_mask = (idx_all >= train_boundary) & (idx_all < valid_boundary)
test_mask = idx_all >= valid_boundary

X_train = X_all[train_mask]
X_valid = X_all[valid_mask]
X_test = X_all[test_mask]

y_train_price = y_price_all[train_mask]
y_valid_price = y_price_all[valid_mask]
y_test_price = y_price_all[test_mask]

idx_train = idx_all[train_mask]
idx_valid = idx_all[valid_mask]
idx_test = idx_all[test_mask]

x_mean = X_train.mean()
x_std = X_train.std() + 1e-8

y_mean = y_train_price.mean()
y_std = y_train_price.std() + 1e-8

X_train_scaled = (X_train - x_mean) / x_std
X_valid_scaled = (X_valid - x_mean) / x_std
X_test_scaled = (X_test - x_mean) / x_std

y_train_scaled = (y_train_price - y_mean) / y_std
y_valid_scaled = (y_valid_price - y_mean) / y_std
y_test_scaled = (y_test_price - y_mean) / y_std

X_train_seq = X_train_scaled[..., None]
X_valid_seq = X_valid_scaled[..., None]
X_test_seq = X_test_scaled[..., None]

last_price_all = X_all[:, -1]
y_dir_all = (y_price_all > last_price_all).astype('float32')

y_train_dir = y_dir_all[train_mask]
y_valid_dir = y_dir_all[valid_mask]
y_test_dir = y_dir_all[test_mask]

X_train.shape, X_valid.shape, X_test.shape


## 6. LSTM for price regression


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, 1)),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1),
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae'],
)

early_stop_lstm = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
)

history_lstm = lstm_model.fit(
    X_train_seq,
    y_train_scaled,
    validation_data=(X_valid_seq, y_valid_scaled),
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[early_stop_lstm],
    shuffle=False,
)

pd.DataFrame(history_lstm.history).plot(title='LSTM training')
plt.show()


In [ ]:
lstm_valid_pred_scaled = lstm_model.predict(X_valid_seq, verbose=0).ravel()
lstm_test_pred_scaled = lstm_model.predict(X_test_seq, verbose=0).ravel()

lstm_valid_pred = lstm_valid_pred_scaled * y_std + y_mean
lstm_test_pred = lstm_test_pred_scaled * y_std + y_mean

lstm_metrics = pd.DataFrame([
    {
        'model': 'LSTM',
        'split': 'valid',
        'MAE': mean_absolute_error(y_valid_price, lstm_valid_pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid_price, lstm_valid_pred)),
    },
    {
        'model': 'LSTM',
        'split': 'test',
        'MAE': mean_absolute_error(y_test_price, lstm_test_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test_price, lstm_test_pred)),
    },
])

lstm_metrics


## 7. CNN for candle pattern classification


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

cnn_classifier = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, 1)),
    tf.keras.layers.Conv1D(32, kernel_size=3, padding='causal', activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(64, kernel_size=3, padding='causal', activation='relu'),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

cnn_classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

early_stop_cnn = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
)

history_cnn = cnn_classifier.fit(
    X_train_seq,
    y_train_dir,
    validation_data=(X_valid_seq, y_valid_dir),
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[early_stop_cnn],
    shuffle=False,
)

pd.DataFrame(history_cnn.history).plot(title='CNN classifier training')
plt.show()


In [ ]:
cnn_valid_prob = cnn_classifier.predict(X_valid_seq, verbose=0).ravel()
cnn_test_prob = cnn_classifier.predict(X_test_seq, verbose=0).ravel()

cnn_valid_pred = (cnn_valid_prob >= 0.5).astype(int)
cnn_test_pred = (cnn_test_prob >= 0.5).astype(int)

cnn_metrics = pd.DataFrame([
    {
        'model': 'CNN classifier',
        'split': 'valid',
        'accuracy': accuracy_score(y_valid_dir, cnn_valid_pred),
        'roc_auc': roc_auc_score(y_valid_dir, cnn_valid_prob),
    },
    {
        'model': 'CNN classifier',
        'split': 'test',
        'accuracy': accuracy_score(y_test_dir, cnn_test_pred),
        'roc_auc': roc_auc_score(y_test_dir, cnn_test_prob),
    },
])

cnn_metrics


## 8. CNN-LSTM for price regression


In [ ]:
tf.keras.utils.set_random_seed(RANDOM_SEED)

cnn_lstm_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(WINDOW, 1)),
    tf.keras.layers.Conv1D(32, kernel_size=3, padding='causal', activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Conv1D(64, kernel_size=3, padding='causal', activation='relu'),
    tf.keras.layers.LSTM(32),
    tf.keras.layers.Dense(1),
])

cnn_lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae'],
)

early_stop_cnn_lstm = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
)

history_cnn_lstm = cnn_lstm_model.fit(
    X_train_seq,
    y_train_scaled,
    validation_data=(X_valid_seq, y_valid_scaled),
    epochs=80,
    batch_size=32,
    verbose=0,
    callbacks=[early_stop_cnn_lstm],
    shuffle=False,
)

pd.DataFrame(history_cnn_lstm.history).plot(title='CNN-LSTM training')
plt.show()


In [ ]:
cnn_lstm_valid_pred_scaled = cnn_lstm_model.predict(X_valid_seq, verbose=0).ravel()
cnn_lstm_test_pred_scaled = cnn_lstm_model.predict(X_test_seq, verbose=0).ravel()

cnn_lstm_valid_pred = cnn_lstm_valid_pred_scaled * y_std + y_mean
cnn_lstm_test_pred = cnn_lstm_test_pred_scaled * y_std + y_mean

regression_comparison = pd.DataFrame([
    {
        'model': 'LSTM',
        'split': 'valid',
        'MAE': mean_absolute_error(y_valid_price, lstm_valid_pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid_price, lstm_valid_pred)),
    },
    {
        'model': 'CNN-LSTM',
        'split': 'valid',
        'MAE': mean_absolute_error(y_valid_price, cnn_lstm_valid_pred),
        'RMSE': np.sqrt(mean_squared_error(y_valid_price, cnn_lstm_valid_pred)),
    },
    {
        'model': 'LSTM',
        'split': 'test',
        'MAE': mean_absolute_error(y_test_price, lstm_test_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test_price, lstm_test_pred)),
    },
    {
        'model': 'CNN-LSTM',
        'split': 'test',
        'MAE': mean_absolute_error(y_test_price, cnn_lstm_test_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test_price, cnn_lstm_test_pred)),
    },
]).sort_values(['split', 'RMSE'])

regression_comparison


## 9. Forecast charts


In [ ]:
test_price_plot = pd.DataFrame({
    'fact_price': y_test_price,
    'lstm_price': lstm_test_pred,
    'cnn_lstm_price': cnn_lstm_test_pred,
}, index=idx_test)

ax = test_price_plot.plot(title=f'Test forecast: ETLN in {FORECAST_HORIZON} trading days')
ax.set_ylabel('RUB')
plt.show()

history_points = 60
history_plot = data['Close'].tail(history_points)
last_window = data['Close'].iloc[-WINDOW:].values.astype('float32')
last_close = float(data['Close'].iloc[-1])
last_date = data.index[-1]
forecast_date = last_date + pd.offsets.BDay(FORECAST_HORIZON)

last_window_scaled = (last_window - x_mean) / x_std

forecast_lstm_scaled = float(lstm_model.predict(last_window_scaled.reshape(1, WINDOW, 1), verbose=0).ravel()[0])
forecast_cnn_lstm_scaled = float(cnn_lstm_model.predict(last_window_scaled.reshape(1, WINDOW, 1), verbose=0).ravel()[0])

forecast_lstm_price = forecast_lstm_scaled * y_std + y_mean
forecast_cnn_lstm_price = forecast_cnn_lstm_scaled * y_std + y_mean

ax = history_plot.plot(
    color='tab:blue',
    linewidth=2,
    label='Current price',
    title=f'ETLN current price and {FORECAST_HORIZON}-day forecast',
)
ax.set_ylabel('RUB')
ax.axvline(last_date, color='gray', linestyle='--', linewidth=1)

ax.plot([last_date, forecast_date], [last_close, forecast_lstm_price], color='tab:orange', linewidth=3, label='LSTM forecast')
ax.plot([last_date, forecast_date], [last_close, forecast_cnn_lstm_price], color='tab:green', linewidth=3, label='CNN-LSTM forecast')

ax.scatter([forecast_date], [forecast_lstm_price], color='tab:orange', s=60)
ax.scatter([forecast_date], [forecast_cnn_lstm_price], color='tab:green', s=60)

ax.legend()
plt.show()

print('Last known close:', round(last_close, 4), 'RUB on', last_date.date())
print(f'LSTM forecast for {FORECAST_HORIZON} days:', round(float(forecast_lstm_price), 4), 'RUB')
print(f'CNN-LSTM forecast for {FORECAST_HORIZON} days:', round(float(forecast_cnn_lstm_price), 4), 'RUB')


## 10. Save outputs


In [ ]:
regression_comparison.to_csv(OUTPUT_DIR / 'metrics_lstm_vs_cnn_lstm_etln.csv', index=False)
cnn_metrics.to_csv(OUTPUT_DIR / 'metrics_cnn_classifier_etln.csv', index=False)
test_price_plot.to_csv(OUTPUT_DIR / 'test_predictions_etln_price.csv')

latest_signal_payload = {
    'figi': FIGI,
    'forecast_horizon': int(FORECAST_HORIZON),
    'signal_date': str(pd.Timestamp(last_date).date()),
    'forecast_target_date': str(pd.Timestamp(forecast_date).date()),
    'current_price': float(last_close),
    'lstm_horizon_price': float(forecast_lstm_price),
    'cnn_lstm_horizon_price': float(forecast_cnn_lstm_price),
}

latest_signal_path = OUTPUT_DIR / f'latest_forecast_signal_etln_h{FORECAST_HORIZON}.json'
latest_signal_path.write_text(json.dumps(latest_signal_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved to:', OUTPUT_DIR.resolve())
print('Signal path:', latest_signal_path.resolve())
